# ⏱️ Reporte Ferroviario de Demora de Tren

**Descripción:** Análisis de boletines de Demora de Tren categorizados por tipo de causa.

**Filtro:** TIPO = 'Demora tren'

**Actualización:** Mensual

---

## 1. Configuración de Rutas

In [1]:
# === CONFIGURACIÓN DE RUTAS ===
from pathlib import Path

# Ruta base del proyecto
BASE_DIR = Path(r"C:\Users\camaro\00-control_trenes\02-boletines")

# Carpetas del proyecto
DATA_DIR = BASE_DIR / "data"
NOTEBOOKS_DIR = BASE_DIR / "notebooks"
REPORTES_DIR = BASE_DIR / "reportes"
CONFIG_DIR = BASE_DIR / "config"
ASSETS_DIR = BASE_DIR / "assets"

# Archivo de datos
ARCHIVO_DATOS = DATA_DIR / "TAC__BOLETINES_REPORTE_MENSUAL_20260401.xlsx"

print(f"📁 Ruta base: {BASE_DIR}")
print(f"📊 Archivo de datos: {ARCHIVO_DATOS}")

📁 Ruta base: C:\Users\camaro\00-control_trenes\02-boletines
📊 Archivo de datos: C:\Users\camaro\00-control_trenes\02-boletines\data\TAC__BOLETINES_REPORTE_MENSUAL_20260401.xlsx


## 2. Importar Librerías

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import json
import re
import unicodedata
import warnings
warnings.filterwarnings('ignore')

# Configuración de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 3. Cargar y Filtrar Datos

In [3]:
# Cargar archivo Excel
df_full = pd.read_excel(ARCHIVO_DATOS, sheet_name='Boletines')
print(f"📋 Total registros cargados: {len(df_full):,}")

# Filtrar solo Demora tren
df = df_full[df_full['TIPO'] == 'Demora tren'].copy()
print(f"⏱️ Registros de Demora Tren: {len(df):,}")
print(f"   ({len(df)/len(df_full)*100:.1f}% del total)")

📋 Total registros cargados: 7,234
⏱️ Registros de Demora Tren: 2,395
   (33.1% del total)


## 4. Preprocesamiento de Datos

In [4]:
# === FUNCIÓN PARA NORMALIZAR TEXTO ===
def normalizar_texto(texto):
    """Convierte a mayúsculas y elimina acentos."""
    if not texto or pd.isna(texto):
        return ''
    texto = str(texto).upper().strip()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    return texto

# === CAMPOS DE FECHA ===
df['FECHA'] = pd.to_datetime(df['FECHA'])
df['FECHA CIERRE'] = pd.to_datetime(df['FECHA CIERRE'])
df['AÑO'] = df['FECHA'].dt.year
df['MES'] = df['FECHA'].dt.month
df['AÑO_MES'] = df['FECHA'].dt.to_period('M').astype(str)

# === CÁLCULO DE DÍAS PARA CIERRE ===
df['DIAS_CIERRE'] = (df['FECHA CIERRE'] - df['FECHA']).dt.days

# === PREPARAR OBSERVACIONES PARA BÚSQUEDA ===
df['OBS_NORM'] = df['OBSERVACIONES'].apply(normalizar_texto)

print("✅ Preprocesamiento completado")
print(f"📅 Período: {df['FECHA'].min().strftime('%d/%m/%Y')} - {df['FECHA'].max().strftime('%d/%m/%Y')}")

✅ Preprocesamiento completado
📅 Período: 28/02/2025 - 31/03/2026


## 5. Definición de Categorías

In [5]:
# === DEFINICIÓN DE CATEGORÍAS DE DEMORA ===

categorias_demora = {
    'PERSONAL': ['RELEVO', 'RELEVOS', 'PERSONAL', 'PNAL', 'POLICIA', 
                 'FALTA DE PERSONAL', 'NEGATIVA AL SERVICIO', 'NO CUENTA CON CONDUCTOR',
                 'CONDUCTOR', 'AYUDANTE', 'JEFE DE TREN', 'DELEGADO', 'PROXIMO EQUIPO DISPONIBLE',
                 'SUPLENCIAS', 'POR MEDIDAS GREMIALES', 'MEDIDAS GREMIALES',
                 'POSTERGADO', 'REVISADOR', 'PARTE DE ENFERMO', 'AUSENTES', 'TECNICO', 
                 'GREMIALES', 'GREMIAL',
                 'CAMBISTA', 'ENFERMO', 'AUSENTE', 'PARTE MEDICA'],
    'MECANICA': ['MECANICA', 'MECANICOS', 'MECANICO', 'TANQUE DE COMBUSTIBLE', 
                 'VAGONES', 'VAGON', 'CONTACTOR P25', 'CONTACTOR', 'P25', 'F/S', 'FRENOS', 'FRENO',
                 'HOMBRE VIVO', 'H.A', 'COMBUSTIBLE', 'ABASTECIDA', 'PROBLEMAS EN LOC',
                 'SE ANULAN', 'MOTORES', 'MOTOR', 'PERDIDA DE AIRE', 'FLUJO DE AIRE',
                 'PERDIDAS DE AIRE', 'SIN ABASTECER', 'PRESION', 'PSI',
                 'SUPERAR PENDIENTE', 'PATINAJE', 'PATINA', 'MANDIBULA', 'FRACCIONAMIENTO',
                 'BOGIE', 'BASTIDOR', 'MANGA', 'VALVULA BP', 'AUXILIA', 'AUXILIAR',
                 'LOCOMOTORA AVERIADA', 'PROBLEMAS EN LOCOMOTORA', 'FALLA LOC',
                 'SUBIR RAMPA', 'TUBERIA', 'ROSCA', 'DE BAJA', 'MEC', 'PENDIENTE', 
                 'ENTREGA DE LOC', 'ESPERA REPUESTO', 'CLC', 'MEVANICA', 'FLUJO', 'REPARACION'],
    'VYO': ['VIA Y OBRA', 'VIA ', 'VIAS', 'OBRA', 'VYO', 'ENTREGA DE VIAS', 'BLOQUEADA',
            'ESPERA RUTA', 'RUTA', 'DESCARRILADO', 'DESCARRILO', 'TORCEDURA DE VIA',
            'CLAUSURA DE VIAS', 'DISPONIBILIDAD DE VIAS', 'V/O', 'CLAUSURA', 'CUADRILLA',
            'GOLPE DE VIA', 'OBSTRUCCION DE VIA', 'ENTREGA DE VIA'],
    'MANIOBRAS': ['INVERTIR LOCOMOTORA', 'DESVIAR TREN', 'DEMORA POR MANIOBRAS', 'MANIOBRAS',
                  'CAMBIAR LOCOMOTORA', 'ESPERA CARGA', 'INVERTIDA', 'SERVICIO MODIFICADO',
                  'A LA ESPERA DE LOCOMOTORA', 'ALISTAMIENTO', 'ESPERA PROGRAMACION',
                  'ESPERAR CRUCE', 'CRUZE', 'CRUCE', 'PROBLEMAS EN CAMBIO', 'FALTA DE LOCOMOTORA',
                  'LIBRADA DE TREN', 'CARGA', 'PENDIENTE A PROGRAMAR', 'QUEDA PENDIENTE',
                  'ESPERA SALIDA', 'AGUARDA', 'FORMACION', 'DETENIDO', 'DETENIDOS',
                  'CANCELADO', 'PROGRAMAR',
                  'DEMORA EN CIRCULACION', 'INGRESO', 'TARDE', 'CORTES', 'INGRESARA', 
                  'RETROCEDERAN', 'MESA GIRATORIA', 'GIRATORIA', 'DEMORADO EN PLAYA', 
                  'BLOQUE EN PLAYA', 'BLOQUEO'],
    'EOT': ['FIN DE TREN', 'EOT', 'FDT'],
    'SISTEMA': ['AUV TELEFONICA', 'STC', 'OBC', 'FALTA DE COMUNICACION', 
                'NO FUNCIONA EL TRANSMISOR', 'TRANSMISOR', 'TECNICO DE SISTEMA',
                'ACTUALIZACION', 'SENAL', 'SATELITE', 'PANTALLA', 'TECLADO', 'MAPEO',
                'COMUNICACION'],
    'ARENA/ARENERO': ['ARENA', 'SIN ARENA', 'ARENERO'],
    'CLIMA': ['LLUEVE', 'LLUVIAS', 'PASTO', 'CORTE DE PASTO', 'INTENSAS LLUVIAS',
              'ALTAS TEMPERATURAS', 'LLOVIZNA', 'ANEGADAS', 'TEMPERATURA']
}

# === COLORES PARA CATEGORÍAS ===
colores_demora = {
    'PERSONAL': '#3498db',
    'MECANICA': '#e74c3c',
    'VYO': '#f39c12',
    'MANIOBRAS': '#9b59b6',
    'EOT': '#1abc9c',
    'SISTEMA': '#e67e22',
    'ARENA/ARENERO': '#34495e',
    'CLIMA': '#27ae60',
    'DEMORA_OTROS': '#95a5a6',
    'SIN_DATOS': '#7f8c8d'
}

orden_categorias = ['PERSONAL', 'MECANICA', 'VYO', 'MANIOBRAS', 'EOT', 'SISTEMA', 'ARENA/ARENERO', 'CLIMA', 'DEMORA_OTROS', 'SIN_DATOS']

print(f"✅ {len(categorias_demora)} categorías definidas + DEMORA_OTROS + SIN_DATOS")

✅ 8 categorías definidas + DEMORA_OTROS + SIN_DATOS


In [6]:
# === NORMALIZAR PALABRAS CLAVE Y CATEGORIZAR ===

categorias_norm = {}
for cat, palabras in categorias_demora.items():
    categorias_norm[cat] = [normalizar_texto(p) for p in palabras]

def categorizar_demora(row):
    """Categoriza una observación de demora."""
    obs = row['OBS_NORM']
    
    # Si es nulo o vacío -> SIN_DATOS
    if not obs or obs.strip() == '' or len(obs.strip()) < 5:
        return 'SIN_DATOS'
    
    # Buscar en categorías
    for categoria, palabras in categorias_norm.items():
        for palabra in palabras:
            if palabra in obs:
                return categoria
    
    # Si tiene datos pero no matchea -> DEMORA_OTROS
    return 'DEMORA_OTROS'

# Aplicar categorización
df['CATEGORIA_DEMORA'] = df.apply(categorizar_demora, axis=1)

# Mostrar distribución
print("\n📊 Distribución por Categoría:")
dist = df['CATEGORIA_DEMORA'].value_counts()
for cat, cant in dist.items():
    pct = cant / len(df) * 100
    print(f"   • {cat}: {cant:,} ({pct:.1f}%)")

sin_datos = len(df[df['CATEGORIA_DEMORA'] == 'SIN_DATOS'])
demora_otros = len(df[df['CATEGORIA_DEMORA'] == 'DEMORA_OTROS'])
clasificados = len(df) - sin_datos - demora_otros
print(f"\n✅ Categorizados: {clasificados:,} ({clasificados/len(df)*100:.1f}%)")
print(f"📭 Sin datos: {sin_datos:,} ({sin_datos/len(df)*100:.1f}%)")
print(f"❓ Demora otros: {demora_otros:,} ({demora_otros/len(df)*100:.1f}%)")


📊 Distribución por Categoría:
   • SIN_DATOS: 1,159 (48.4%)
   • PERSONAL: 432 (18.0%)
   • MECANICA: 302 (12.6%)
   • VYO: 162 (6.8%)
   • DEMORA_OTROS: 149 (6.2%)
   • MANIOBRAS: 96 (4.0%)
   • EOT: 45 (1.9%)
   • SISTEMA: 43 (1.8%)
   • ARENA/ARENERO: 5 (0.2%)
   • CLIMA: 2 (0.1%)

✅ Categorizados: 1,087 (45.4%)
📭 Sin datos: 1,159 (48.4%)
❓ Demora otros: 149 (6.2%)


## 6. Cálculos para el Reporte

In [7]:
# === CÁLCULOS GENERALES ===
total_registros = len(df)
fecha_min = df['FECHA'].min().strftime('%d/%m/%Y')
fecha_max = df['FECHA'].max().strftime('%d/%m/%Y')
fecha_generacion = datetime.now().strftime('%d/%m/%Y %H:%M')
meses_ordenados = sorted(df['AÑO_MES'].unique())
lineas_disponibles = sorted(df['LINEA'].unique().tolist())
anios_disponibles = sorted(df['AÑO'].unique().tolist())

# Tiempo promedio de cierre
tiempo_promedio_cierre = df[df['ESTADO'] == 'Cerrado']['DIAS_CIERRE'].mean()
if pd.isna(tiempo_promedio_cierre):
    tiempo_promedio_cierre = 0

# === KPIs POR CATEGORÍA ===
kpis_categoria = df['CATEGORIA_DEMORA'].value_counts().reset_index()
kpis_categoria.columns = ['Categoría', 'Cantidad']
kpis_categoria['Porcentaje'] = (kpis_categoria['Cantidad'] / total_registros * 100).round(2)
kpis_categoria['Categoría'] = pd.Categorical(kpis_categoria['Categoría'], categories=orden_categorias, ordered=True)
kpis_categoria = kpis_categoria.sort_values('Categoría').reset_index(drop=True)

# === ESTADO DE BOLETINES ===
estado_boletines = df['ESTADO'].value_counts().reset_index()
estado_boletines.columns = ['Estado', 'Cantidad']
estado_boletines['Porcentaje'] = (estado_boletines['Cantidad'] / total_registros * 100).round(2)

# === DISTRIBUCIÓN POR LÍNEA ===
dist_linea = df['LINEA'].value_counts().reset_index()
dist_linea.columns = ['Línea', 'Cantidad']
dist_linea['Porcentaje'] = (dist_linea['Cantidad'] / total_registros * 100).round(2)

# === CATEGORÍA x LÍNEA ===
cat_linea = pd.crosstab(df['LINEA'], df['CATEGORIA_DEMORA'])
cat_linea = cat_linea.reindex(columns=orden_categorias, fill_value=0)

print("✅ Cálculos básicos completados")

✅ Cálculos básicos completados


In [8]:
# === DATOS PARA GRÁFICO TENDENCIA POR CATEGORÍA ===

datos_tendencia = {}

# Total (Todas las líneas)
tendencia_total = df.groupby(['AÑO_MES', 'CATEGORIA_DEMORA']).size().unstack(fill_value=0)
tendencia_total = tendencia_total.reindex(columns=orden_categorias, fill_value=0)
tendencia_total = tendencia_total.reindex(meses_ordenados, fill_value=0)
total_por_mes = df.groupby('AÑO_MES').size().reindex(meses_ordenados, fill_value=0)

datos_tendencia['Todas'] = {
    'meses': meses_ordenados,
    'categorias': {cat: tendencia_total[cat].tolist() for cat in orden_categorias},
    'total': total_por_mes.tolist()
}

# Por cada línea
for linea in lineas_disponibles:
    df_linea = df[df['LINEA'] == linea]
    tend_linea = df_linea.groupby(['AÑO_MES', 'CATEGORIA_DEMORA']).size().unstack(fill_value=0)
    tend_linea = tend_linea.reindex(columns=orden_categorias, fill_value=0)
    tend_linea = tend_linea.reindex(meses_ordenados, fill_value=0)
    total_linea = df_linea.groupby('AÑO_MES').size().reindex(meses_ordenados, fill_value=0)
    
    datos_tendencia[linea] = {
        'meses': meses_ordenados,
        'categorias': {cat: tend_linea[cat].tolist() for cat in orden_categorias},
        'total': total_linea.tolist()
    }

datos_tendencia_json = json.dumps(datos_tendencia)
colores_demora_json = json.dumps(colores_demora)

print("✅ Datos para gráfico de tendencia preparados")

✅ Datos para gráfico de tendencia preparados


In [9]:
# === DATOS PARA GRÁFICO UP (con filtros Línea + Año + Mes) ===

datos_up = {}
nombres_meses = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
                 7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}

# Estructura: datos_up[linea][año][mes] = {ups, cantidades, porcentajes}
# linea: 'Todas' o nombre de línea
# año: 'Todos' o número de año
# mes: 'Todos' o número de mes (1-12)

def calcular_up(df_filtrado):
    if len(df_filtrado) == 0:
        return {'ups': [], 'cantidades': [], 'porcentajes': []}
    up_data = df_filtrado['UP'].value_counts().head(12)
    total = len(df_filtrado)
    return {
        'ups': up_data.index.tolist(),
        'cantidades': up_data.values.tolist(),
        'porcentajes': (up_data.values / total * 100).round(2).tolist()
    }

# Todas las combinaciones
lineas_para_up = ['Todas'] + lineas_disponibles
anios_para_up = ['Todos'] + [str(a) for a in anios_disponibles]
meses_para_up = ['Todos'] + list(range(1, 13))

for linea in lineas_para_up:
    datos_up[linea] = {}
    df_linea = df if linea == 'Todas' else df[df['LINEA'] == linea]
    
    for anio in anios_para_up:
        datos_up[linea][anio] = {}
        if anio == 'Todos':
            df_anio = df_linea
        else:
            df_anio = df_linea[df_linea['AÑO'] == int(anio)]
        
        for mes in meses_para_up:
            if mes == 'Todos':
                df_mes = df_anio
            else:
                df_mes = df_anio[df_anio['MES'] == mes]
            
            key_mes = 'Todos' if mes == 'Todos' else str(mes)
            datos_up[linea][anio][key_mes] = calcular_up(df_mes)

datos_up_json = json.dumps(datos_up)
anios_disponibles_json = json.dumps([str(a) for a in anios_disponibles])

print("✅ Datos para gráfico de UP preparados (con filtros Línea + Año + Mes)")

✅ Datos para gráfico de UP preparados (con filtros Línea + Año + Mes)


In [10]:
# === DATOS PARA GRÁFICO ESTADO POR MES ===

orden_estados = ['Abierto', 'Seguimiento', 'Cerrado']
colores_estados = {
    'Abierto': '#ef4444',
    'Seguimiento': '#f59e0b',
    'Cerrado': '#10b981'
}

datos_estado = {}

# Todas las líneas
estado_por_mes = df.groupby(['AÑO_MES', 'ESTADO']).size().unstack(fill_value=0)
estado_por_mes = estado_por_mes.reindex(columns=orden_estados, fill_value=0)
estado_por_mes = estado_por_mes.reindex(meses_ordenados, fill_value=0)

total_mes = estado_por_mes.sum(axis=1)
pct_cierre = (estado_por_mes['Cerrado'] / total_mes * 100).round(2).fillna(0)

datos_estado['Todas'] = {
    'meses': meses_ordenados,
    'estados': {estado: estado_por_mes[estado].tolist() for estado in orden_estados},
    'pct_cierre': pct_cierre.tolist()
}

# Por línea
for linea in lineas_disponibles:
    df_linea = df[df['LINEA'] == linea]
    estado_linea = df_linea.groupby(['AÑO_MES', 'ESTADO']).size().unstack(fill_value=0)
    estado_linea = estado_linea.reindex(columns=orden_estados, fill_value=0)
    estado_linea = estado_linea.reindex(meses_ordenados, fill_value=0)
    
    total_mes_linea = estado_linea.sum(axis=1)
    pct_cierre_linea = (estado_linea['Cerrado'] / total_mes_linea * 100).round(2).fillna(0)
    
    datos_estado[linea] = {
        'meses': meses_ordenados,
        'estados': {estado: estado_linea[estado].tolist() for estado in orden_estados},
        'pct_cierre': pct_cierre_linea.tolist()
    }

datos_estado_json = json.dumps(datos_estado)
colores_estados_json = json.dumps(colores_estados)

print("✅ Datos para gráfico de estado preparados")

✅ Datos para gráfico de estado preparados


## 7. Generación del HTML

In [11]:
# === GENERAR HTML COMPLETO ===

# Calcular % clasificado (excluyendo SIN_DATOS y DEMORA_OTROS)
sin_datos_val = kpis_categoria[kpis_categoria['Categoría']=='SIN_DATOS']['Cantidad'].values[0] if 'SIN_DATOS' in kpis_categoria['Categoría'].values else 0
demora_otros_val = kpis_categoria[kpis_categoria['Categoría']=='DEMORA_OTROS']['Cantidad'].values[0] if 'DEMORA_OTROS' in kpis_categoria['Categoría'].values else 0
pct_clasificado = ((total_registros - sin_datos_val - demora_otros_val) / total_registros * 100)

# Generar filas de KPIs por categoría
kpi_cards_html = ''
for cat in orden_categorias:
    css_class = cat.lower().replace('/', '-').replace(' ', '-')
    cant = int(kpis_categoria[kpis_categoria['Categoría']==cat]['Cantidad'].values[0]) if cat in kpis_categoria['Categoría'].values else 0
    pct = kpis_categoria[kpis_categoria['Categoría']==cat]['Porcentaje'].values[0] if cat in kpis_categoria['Categoría'].values else 0
    kpi_cards_html += f'''<div class="kpi-card {css_class}"><div class="kpi-title">{cat}</div><div class="kpi-value">{cant:,}</div><div class="kpi-percent">{pct:.1f}%</div></div>\n'''

# Generar filas de tabla estado
estado_rows_html = ''
for _, row in estado_boletines.iterrows():
    estado_rows_html += f'''<tr><td><span class="estado-{row['Estado'].lower()}">{row['Estado']}</span></td><td>{int(row['Cantidad']):,}</td><td>{row['Porcentaje']:.1f}%</td></tr>\n'''

# Generar filas de tabla línea
linea_rows_html = ''
for _, row in dist_linea.iterrows():
    linea_rows_html += f'''<tr><td>{row['Línea']}</td><td>{int(row['Cantidad']):,}</td><td>{row['Porcentaje']:.1f}%</td></tr>\n'''

# Generar opciones de línea
linea_options_html = '\n'.join([f'<option value="{linea}">{linea}</option>' for linea in lineas_disponibles])

# Generar opciones de año para UP
anio_options_html = '\n'.join([f'<option value="{anio}">{anio}</option>' for anio in anios_disponibles])

# Generar checkboxes de categorías
checkbox_html = ''
for cat in orden_categorias:
    chk_id = cat.lower().replace('/', '_').replace(' ', '_')
    checked = 'checked' if cat != 'SIN_DATOS' else ''
    color = colores_demora[cat]
    checkbox_html += f'''<div class="checkbox-item"><input type="checkbox" id="chk_{chk_id}" {checked} onchange="toggleCategoria('{cat}')"><span class="color-dot" style="background: {color};"></span><label for="chk_{chk_id}">{cat[:8]}</label></div>\n'''

# Generar tabla categoría x línea
cat_linea_header = '<th>Línea</th>' + ''.join([f'<th style="font-size:0.6rem;">{cat[:6]}</th>' for cat in orden_categorias]) + '<th>Total</th>'
cat_linea_rows = ''
for linea in cat_linea.index:
    row_html = f'<td>{linea}</td>'
    for cat in orden_categorias:
        val = int(cat_linea.loc[linea, cat]) if cat in cat_linea.columns else 0
        row_html += f'<td>{val:,}</td>'
    row_html += f'<td><strong>{int(cat_linea.loc[linea].sum()):,}</strong></td>'
    cat_linea_rows += f'<tr>{row_html}</tr>\n'

print("✅ Componentes HTML preparados")

✅ Componentes HTML preparados


In [12]:
html_content = f'''<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Reporte Ferroviario de Demora de Tren</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
    <style>
        :root {{ --bg-primary: #0a0f1c; --bg-secondary: #111827; --bg-card: #1a2234; --bg-card-hover: #212d45; --text-primary: #f1f5f9; --text-secondary: #94a3b8; --text-muted: #64748b; --border-color: #2d3a4f; --gradient-1: linear-gradient(135deg, #f39c12 0%, #e74c3c 100%); --shadow-lg: 0 25px 50px -12px rgba(0, 0, 0, 0.5); }}
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: "Outfit", sans-serif; background: var(--bg-primary); color: var(--text-primary); line-height: 1.6; min-height: 100vh; }}
        .container {{ max-width: 1400px; margin: 0 auto; padding: 2rem; }}
        .header {{ background: var(--bg-secondary); border-bottom: 1px solid var(--border-color); padding: 2rem 0; margin-bottom: 2rem; }}
        .header-content {{ max-width: 1400px; margin: 0 auto; padding: 0 2rem; display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 1rem; }}
        .header h1 {{ font-size: 2rem; font-weight: 700; background: var(--gradient-1); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; }}
        .header-meta {{ display: flex; gap: 2rem; font-size: 0.9rem; color: var(--text-secondary); }}
        .header-meta span {{ display: flex; align-items: center; gap: 0.5rem; }}
        .info-cards {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 1rem; margin-bottom: 2rem; }}
        .info-card {{ background: var(--bg-card); border: 1px solid var(--border-color); border-radius: 12px; padding: 1.5rem; text-align: center; transition: transform 0.2s, box-shadow 0.2s; }}
        .info-card:hover {{ transform: translateY(-2px); box-shadow: var(--shadow-lg); }}
        .info-card .label {{ font-size: 0.85rem; color: var(--text-muted); text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 0.5rem; }}
        .info-card .value {{ font-size: 1.75rem; font-weight: 700; font-family: "JetBrains Mono", monospace; }}
        .info-card.orange .value {{ color: #f39c12; }}
        .info-card.blue .value {{ color: #3498db; }}
        .info-card.red .value {{ color: #e74c3c; }}
        .info-card.green .value {{ color: #10b981; }}
        .section {{ background: var(--bg-card); border: 1px solid var(--border-color); border-radius: 16px; padding: 1.5rem; margin-bottom: 1.5rem; }}
        .section-title {{ font-size: 1.1rem; font-weight: 600; color: var(--text-primary); margin-bottom: 1.25rem; padding-bottom: 0.75rem; border-bottom: 1px solid var(--border-color); display: flex; align-items: center; gap: 0.5rem; }}
        .kpi-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 0.75rem; }}
        .kpi-card {{ background: var(--bg-secondary); border-radius: 10px; padding: 1rem; text-align: center; border-left: 4px solid #3498db; }}
        .kpi-card.personal {{ border-left-color: #3498db; }}
        .kpi-card.mecanica {{ border-left-color: #e74c3c; }}
        .kpi-card.vyo {{ border-left-color: #f39c12; }}
        .kpi-card.maniobras {{ border-left-color: #9b59b6; }}
        .kpi-card.eot {{ border-left-color: #1abc9c; }}
        .kpi-card.sistema {{ border-left-color: #e67e22; }}
        .kpi-card.arena-arenero {{ border-left-color: #34495e; }}
        .kpi-card.clima {{ border-left-color: #27ae60; }}
        .kpi-card.demora_otros {{ border-left-color: #95a5a6; }}
        .kpi-card.sin_datos {{ border-left-color: #7f8c8d; }}
        .kpi-card .kpi-title {{ font-size: 0.65rem; color: var(--text-muted); margin-bottom: 0.25rem; text-transform: uppercase; }}
        .kpi-card .kpi-value {{ font-size: 1.1rem; font-weight: 700; font-family: "JetBrains Mono", monospace; color: var(--text-primary); }}
        .kpi-card .kpi-percent {{ font-size: 0.75rem; color: var(--text-secondary); }}
        .table-container {{ overflow-x: auto; }}
        table {{ width: 100%; border-collapse: collapse; font-size: 0.9rem; }}
        th, td {{ padding: 0.875rem 1rem; text-align: left; }}
        th {{ background: var(--bg-secondary); color: var(--text-secondary); font-weight: 500; text-transform: uppercase; font-size: 0.75rem; letter-spacing: 0.5px; }}
        tr {{ border-bottom: 1px solid var(--border-color); }}
        tr:hover {{ background: var(--bg-card-hover); }}
        .grid-2 {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(400px, 1fr)); gap: 1.5rem; }}
        .chart-container {{ position: relative; height: 350px; width: 100%; }}
        .chart-container-lg {{ position: relative; height: 400px; width: 100%; }}
        .estado-cerrado {{ color: #10b981; }}
        .estado-abierto {{ color: #ef4444; }}
        .estado-seguimiento {{ color: #f59e0b; }}
        .footer {{ text-align: center; padding: 2rem; color: var(--text-muted); font-size: 0.85rem; border-top: 1px solid var(--border-color); margin-top: 2rem; }}
        .chart-controls {{ display: flex; flex-wrap: wrap; gap: 1rem; margin-bottom: 1.5rem; align-items: center; }}
        .chart-controls select {{ background: var(--bg-secondary); border: 1px solid var(--border-color); color: var(--text-primary); padding: 0.5rem 1rem; border-radius: 8px; font-family: "Outfit", sans-serif; font-size: 0.9rem; cursor: pointer; min-width: 120px; }}
        .chart-controls select:focus {{ outline: none; border-color: #f39c12; }}
        .filter-label {{ font-size: 0.85rem; color: var(--text-muted); text-transform: uppercase; letter-spacing: 0.5px; }}
        .checkbox-group {{ display: flex; flex-wrap: wrap; gap: 0.4rem; align-items: center; }}
        .checkbox-item {{ display: flex; align-items: center; gap: 0.25rem; cursor: pointer; padding: 0.25rem 0.5rem; border-radius: 6px; background: var(--bg-secondary); border: 1px solid var(--border-color); transition: all 0.2s; }}
        .checkbox-item:hover {{ border-color: #f39c12; }}
        .checkbox-item input {{ cursor: pointer; width: 12px; height: 12px; }}
        .checkbox-item label {{ cursor: pointer; font-size: 0.7rem; color: var(--text-secondary); white-space: nowrap; }}
        .checkbox-item .color-dot {{ width: 8px; height: 8px; border-radius: 50%; }}
        @media (max-width: 768px) {{ .container {{ padding: 1rem; }} .header-content {{ flex-direction: column; text-align: center; }} .grid-2 {{ grid-template-columns: 1fr; }} .chart-controls {{ flex-direction: column; align-items: flex-start; }} }}
    </style>
</head>
<body>
    <header class="header">
        <div class="header-content">
            <h1>⏱️ Reporte Ferroviario de Demora de Tren</h1>
            <div class="header-meta">
                <span>📅 Período: {fecha_min} - {fecha_max}</span>
                <span>🕐 Generado: {fecha_generacion}</span>
            </div>
        </div>
    </header>
    <div class="container">
        <div class="info-cards">
            <div class="info-card orange"><div class="label">Total Demoras</div><div class="value">{total_registros:,}</div></div>
            <div class="info-card blue"><div class="label">Tiempo Prom. Cierre</div><div class="value">{tiempo_promedio_cierre:.1f} días</div></div>
            <div class="info-card green"><div class="label">% Clasificado</div><div class="value">{pct_clasificado:.1f}%</div></div>
            <div class="info-card red"><div class="label">Sin Datos</div><div class="value">{int(sin_datos_val):,}</div></div>
        </div>
        <div class="section">
            <h2 class="section-title">📊 Distribución por Categoría de Demora</h2>
            <div class="kpi-grid">
{kpi_cards_html}
            </div>
        </div>
        <div class="grid-2">
            <div class="section">
                <h2 class="section-title">📋 Estado de Boletines</h2>
                <div class="table-container"><table><thead><tr><th>Estado</th><th>Cantidad</th><th>Porcentaje</th></tr></thead><tbody>
{estado_rows_html}
                </tbody></table></div>
            </div>
            <div class="section">
                <h2 class="section-title">🚂 Distribución por Línea</h2>
                <div class="table-container"><table><thead><tr><th>Línea</th><th>Cantidad</th><th>Porcentaje</th></tr></thead><tbody>
{linea_rows_html}
                </tbody></table></div>
            </div>
        </div>
        <div class="section">
            <h2 class="section-title">📈 Tendencia Mensual por Categoría</h2>
            <div class="chart-controls">
                <div><span class="filter-label">Línea: </span>
                    <select id="lineaTendenciaSelect" onchange="actualizarGraficoTendencia()">
                        <option value="Todas">Todas las Líneas</option>
{linea_options_html}
                    </select>
                </div>
                <div class="checkbox-group">
                    <span class="filter-label">Mostrar: </span>
                    <div class="checkbox-item"><input type="checkbox" id="chk_total" checked onchange="toggleCategoria('Total')"><span class="color-dot" style="background: #ffffff;"></span><label for="chk_total">Total</label></div>
{checkbox_html}
                </div>
            </div>
            <div class="chart-container"><canvas id="tendenciaChart"></canvas></div>
        </div>
        <div class="section">
            <h2 class="section-title">🏢 Distribución por Unidad de Producción (UP)</h2>
            <div class="chart-controls">
                <div><span class="filter-label">Línea: </span>
                    <select id="lineaUpSelect" onchange="actualizarGraficoUp()">
                        <option value="Todas">Todas las Líneas</option>
{linea_options_html}
                    </select>
                </div>
                <div><span class="filter-label">Año: </span>
                    <select id="anioUpSelect" onchange="actualizarGraficoUp()">
                        <option value="Todos">Todos</option>
{anio_options_html}
                    </select>
                </div>
                <div><span class="filter-label">Mes: </span>
                    <select id="mesUpSelect" onchange="actualizarGraficoUp()">
                        <option value="Todos">Todos</option>
                        <option value="1">Enero</option>
                        <option value="2">Febrero</option>
                        <option value="3">Marzo</option>
                        <option value="4">Abril</option>
                        <option value="5">Mayo</option>
                        <option value="6">Junio</option>
                        <option value="7">Julio</option>
                        <option value="8">Agosto</option>
                        <option value="9">Septiembre</option>
                        <option value="10">Octubre</option>
                        <option value="11">Noviembre</option>
                        <option value="12">Diciembre</option>
                    </select>
                </div>
            </div>
            <div class="chart-container-lg"><canvas id="upChart"></canvas></div>
        </div>
        <div class="section">
            <h2 class="section-title">📊 Categoría de Demora por Línea</h2>
            <div class="table-container"><table><thead><tr>{cat_linea_header}</tr></thead><tbody>
{cat_linea_rows}
            </tbody></table></div>
        </div>
        <div class="section">
            <h2 class="section-title">📊 Estado de Boletines por Mes</h2>
            <div class="chart-controls">
                <div><span class="filter-label">Línea: </span>
                    <select id="lineaEstadoSelect" onchange="actualizarGraficoEstado()">
                        <option value="Todas">Todas las Líneas</option>
{linea_options_html}
                    </select>
                </div>
            </div>
            <div class="chart-container-lg"><canvas id="estadoChart"></canvas></div>
        </div>
        <footer class="footer"><p>Reporte Ferroviario de Demora de Tren | Generado por carlos amaro | camaro@bcyl.com.ar</p></footer>
    </div>
    <script>
        const datosTendencia = {datos_tendencia_json};
        const coloresDemora = {colores_demora_json};
        const ordenCategorias = {json.dumps(orden_categorias)};
        const datosUp = {datos_up_json};
        const datosEstado = {datos_estado_json};
        const coloresEstados = {colores_estados_json};
        const ordenEstados = ['Abierto', 'Seguimiento', 'Cerrado'];
        let categoriasVisibles = {{ 'Total': true }};
        ordenCategorias.forEach(cat => {{ categoriasVisibles[cat] = (cat !== 'SIN_DATOS'); }});
        let chartTendencia = null;
        function crearGraficoTendencia() {{
            const ctx = document.getElementById('tendenciaChart').getContext('2d');
            const lineaSeleccionada = document.getElementById('lineaTendenciaSelect').value;
            const datos = datosTendencia[lineaSeleccionada];
            const datasets = [{{ label: 'Total', data: datos.total, borderColor: '#ffffff', backgroundColor: 'rgba(255, 255, 255, 0.1)', borderWidth: 3, fill: false, tension: 0.3, pointBackgroundColor: '#ffffff', pointBorderColor: '#0a0f1c', pointBorderWidth: 2, pointRadius: 5, pointHoverRadius: 7, hidden: !categoriasVisibles['Total'] }}];
            ordenCategorias.forEach(cat => {{ datasets.push({{ label: cat, data: datos.categorias[cat], borderColor: coloresDemora[cat], backgroundColor: coloresDemora[cat] + '20', borderWidth: 2, fill: false, tension: 0.3, pointBackgroundColor: coloresDemora[cat], pointBorderColor: '#fff', pointBorderWidth: 1, pointRadius: 3, pointHoverRadius: 5, hidden: !categoriasVisibles[cat] }}); }});
            if (chartTendencia) chartTendencia.destroy();
            chartTendencia = new Chart(ctx, {{ type: 'line', data: {{ labels: datos.meses, datasets: datasets }}, options: {{ responsive: true, maintainAspectRatio: false, interaction: {{ mode: 'index', intersect: false }}, plugins: {{ legend: {{ display: false }}, tooltip: {{ backgroundColor: 'rgba(17, 24, 39, 0.95)', titleColor: '#f1f5f9', bodyColor: '#94a3b8', borderColor: '#2d3a4f', borderWidth: 1, padding: 12 }} }}, scales: {{ y: {{ beginAtZero: true, grid: {{ color: 'rgba(255,255,255,0.08)' }}, ticks: {{ color: '#94a3b8' }} }}, x: {{ grid: {{ display: false }}, ticks: {{ color: '#94a3b8', maxRotation: 45 }} }} }} }} }});
        }}
        function actualizarGraficoTendencia() {{ crearGraficoTendencia(); }}
        function toggleCategoria(cat) {{ categoriasVisibles[cat] = !categoriasVisibles[cat]; crearGraficoTendencia(); }}
        let chartUp = null;
        function crearGraficoUp() {{
            const ctx = document.getElementById('upChart').getContext('2d');
            const lineaSeleccionada = document.getElementById('lineaUpSelect').value;
            const anioSeleccionado = document.getElementById('anioUpSelect').value;
            const mesSeleccionado = document.getElementById('mesUpSelect').value;
            const datos = datosUp[lineaSeleccionada][anioSeleccionado][mesSeleccionado];
            if (chartUp) chartUp.destroy();
            if (!datos || datos.ups.length === 0) {{
                chartUp = new Chart(ctx, {{ type: 'bar', data: {{ labels: ['Sin datos'], datasets: [{{ label: 'Cantidad', data: [0], backgroundColor: 'rgba(100,100,100,0.3)' }}] }}, options: {{ responsive: true, maintainAspectRatio: false, indexAxis: 'y', plugins: {{ legend: {{ display: false }} }} }} }});
                return;
            }}
            chartUp = new Chart(ctx, {{ type: 'bar', data: {{ labels: datos.ups, datasets: [{{ label: 'Cantidad', data: datos.cantidades, backgroundColor: 'rgba(243, 156, 18, 0.7)', borderColor: '#f39c12', borderWidth: 1, borderRadius: 4, yAxisID: 'y' }}, {{ label: 'Porcentaje', data: datos.porcentajes, type: 'line', borderColor: '#e74c3c', borderWidth: 3, fill: false, tension: 0.3, pointBackgroundColor: '#e74c3c', pointRadius: 5, yAxisID: 'y1' }}] }}, options: {{ responsive: true, maintainAspectRatio: false, indexAxis: 'y', plugins: {{ legend: {{ display: true, position: 'top', labels: {{ color: '#94a3b8' }} }}, tooltip: {{ callbacks: {{ label: function(ctx) {{ if (ctx.dataset.label === 'Porcentaje') return ctx.dataset.label + ': ' + ctx.parsed.x.toFixed(1) + '%'; return ctx.dataset.label + ': ' + ctx.parsed.x.toLocaleString(); }} }} }} }}, scales: {{ x: {{ beginAtZero: true, grid: {{ color: 'rgba(255,255,255,0.08)' }}, ticks: {{ color: '#94a3b8' }} }}, x1: {{ position: 'top', beginAtZero: true, grid: {{ drawOnChartArea: false }}, ticks: {{ color: '#e74c3c', callback: v => v + '%' }} }}, y: {{ grid: {{ display: false }}, ticks: {{ color: '#94a3b8' }} }} }} }} }});
        }}
        function actualizarGraficoUp() {{ crearGraficoUp(); }}
        let chartEstado = null;
        function crearGraficoEstado() {{
            const ctx = document.getElementById('estadoChart').getContext('2d');
            const lineaSeleccionada = document.getElementById('lineaEstadoSelect').value;
            const datos = datosEstado[lineaSeleccionada];
            if (chartEstado) chartEstado.destroy();
            const datasets = ordenEstados.map(estado => ({{ label: estado, data: datos.estados[estado], backgroundColor: coloresEstados[estado] + 'CC', borderColor: coloresEstados[estado], borderWidth: 1, borderRadius: 4, yAxisID: 'y' }}));
            datasets.push({{ label: '% Cierre', data: datos.pct_cierre, type: 'line', borderColor: '#8b5cf6', borderWidth: 3, fill: false, tension: 0.3, pointBackgroundColor: '#8b5cf6', pointRadius: 5, yAxisID: 'y1' }});
            chartEstado = new Chart(ctx, {{ type: 'bar', data: {{ labels: datos.meses, datasets: datasets }}, options: {{ responsive: true, maintainAspectRatio: false, plugins: {{ legend: {{ display: true, position: 'top', labels: {{ color: '#94a3b8' }} }}, tooltip: {{ callbacks: {{ label: function(ctx) {{ if (ctx.dataset.label === '% Cierre') return ctx.dataset.label + ': ' + ctx.parsed.y.toFixed(1) + '%'; return ctx.dataset.label + ': ' + ctx.parsed.y.toLocaleString(); }} }} }} }}, scales: {{ y: {{ beginAtZero: true, stacked: true, grid: {{ color: 'rgba(255,255,255,0.08)' }}, ticks: {{ color: '#94a3b8' }}, title: {{ display: true, text: 'Cantidad', color: '#94a3b8' }} }}, y1: {{ position: 'right', beginAtZero: true, max: 100, grid: {{ drawOnChartArea: false }}, ticks: {{ color: '#8b5cf6', callback: v => v + '%' }}, title: {{ display: true, text: '% Cierre', color: '#8b5cf6' }} }}, x: {{ stacked: true, grid: {{ display: false }}, ticks: {{ color: '#94a3b8', maxRotation: 45 }} }} }} }} }});
        }}
        function actualizarGraficoEstado() {{ crearGraficoEstado(); }}
        document.addEventListener('DOMContentLoaded', function() {{ crearGraficoTendencia(); crearGraficoUp(); crearGraficoEstado(); }});
    </script>
</body>
</html>'''

print("✅ HTML generado exitosamente")

✅ HTML generado exitosamente


## 8. Exportar Reporte HTML

In [13]:
# === EXPORTAR REPORTE ===

# Nombre del archivo con año y mes del último dato
ultimo_periodo = df['FECHA'].max().strftime('%Y%m')
nombre_reporte = f"reporte_ferroviario_demora_tren_{ultimo_periodo}.html"
ruta_reporte = REPORTES_DIR / nombre_reporte

# Crear carpeta si no existe
REPORTES_DIR.mkdir(parents=True, exist_ok=True)

# Guardar HTML
with open(ruta_reporte, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"✅ Reporte exportado exitosamente")
print(f"📁 Ubicación: {ruta_reporte}")
print(f"\n💡 Para abrir el reporte, ejecuta:")
print(f"   import webbrowser; webbrowser.open(r'{ruta_reporte}')")

✅ Reporte exportado exitosamente
📁 Ubicación: C:\Users\camaro\00-control_trenes\02-boletines\reportes\reporte_ferroviario_demora_tren_202603.html

💡 Para abrir el reporte, ejecuta:
   import webbrowser; webbrowser.open(r'C:\Users\camaro\00-control_trenes\02-boletines\reportes\reporte_ferroviario_demora_tren_202603.html')


In [14]:
# === ABRIR REPORTE EN NAVEGADOR (opcional) ===
import webbrowser
webbrowser.open(str(ruta_reporte))

True